In [14]:
import os
import numpy as np
import json
from models.hybridgnet_se_resnext_dual import HybridDual
from models.hybridgnet_se_resnext import Hybrid
from models.utils import load_config
import torch
import matplotlib.pyplot as plt
import cv2
import matplotlib.gridspec as gridspec

In [15]:
DATASET = '../Dataset/CAMUS/Landmarks_3_10'
NAME = 'CAMUS_atlas_seg_NN'

hyperparameters = json.load(open("../Training/%s/hyperparameters.json"%NAME))
config, D_t, U_t, A_t = load_config(DATASET, hyperparameters)
config['resume'] = "../Training/%s/%s.pth"%(NAME, NAME)
config['raster_as_input'] = True

if config['use_dual']:
    model = HybridDual(config, D_t, U_t, A_t).to(config['device'])
else:
    model = Hybrid(config, D_t, U_t, A_t).to(config['device'])

print("Image Encoder filters", model.encoder.filters + [model.encoder.filters[-1]])
print("Bottleneck latents", model.encoder.latents)
print("Graph convolutional filters", config['filters'][::-1])

if config['resume']:
    model.load_checkpoint(config['resume'], config['device'])

model.eval()
print("Model loaded from", config['resume'])

Configuration
database_path : ../CAMUS/database_nifti
output_path : ../Dataset/CAMUS/Landmarks_3_10
scale_factor : 0.1
resolutions : ['full', 'half', 'quarter']
image_types : ['2CH', '4CH']
organs : ['1', '2', '3']
organ_names : ['LV Endo', 'LV Epi', 'LA']
inputsize : 512
flip_h : True
flip_v : False
rotate : True
transpose : False

Loading adjacency matrices ../Dataset/CAMUS/Landmarks_3_10/NonNaive/adj_full_block_diagonal.npy
Image Encoder filters [16, 32, 64, 128, 256, 256]
Bottleneck latents 64
Graph convolutional filters [32, 32, 24, 24, 16, 16, 8, 2]
Model loaded from ../Training/CAMUS_atlas_seg_NN/CAMUS_atlas_seg_NN.pth


In [16]:
if not config["naive"]:
    organ_id = np.load("%s/NonNaive/adj_full_organ_id.npy" % DATASET)[:,0]
    organs = config['organs']
    organ_names = config['organ_names']
    organ_dict = {organs[i]: organ_names[i] for i in range(len(organ_names))}

    with open(f"{DATASET}/NonNaive/organ_order_full.json", "r") as f:
        circ_organ_order = json.load(f)
else:
    organ_id = np.load("%s/Naive/adj_full_organ_id.npy" % DATASET)[:,0]
    organs = config['organs']
    organ_names = config['organ_names']
    organ_dict = {organs[i]: organ_names[i] for i in range(len(organ_names))}
    
# Define colors for visualization
organ_colors = {
    'LV Endo': 'red',
    'LV Epi': 'green', 
    'LA': 'blue'
}


In [17]:
import pandas as pd 
from medpy.metric.binary import dc as dice_score

images_path = "/home/ngaggion/Documents/Mask2Graph/CAMUS/nnUNet_raw/predTs"
images = sorted([f for f in os.listdir(images_path) if f.endswith('.png')])
images = [os.path.join(images_path, f) for f in images]

df = pd.DataFrame(columns=['Image', 'Organ', 'Dice', 'Unified'])

for i, image_path in enumerate(images):
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    # scale is pad to square then resize to 512x512
    # padding to square
    h, w = image.shape
    if h > w:
        pad = (h - w) // 2
        image = cv2.copyMakeBorder(image, 0, 0, pad, pad, cv2.BORDER_CONSTANT, value=0)
    elif w > h:
        pad = (w - h) // 2
        image = cv2.copyMakeBorder(image, pad, pad, 0, 0, cv2.BORDER_CONSTANT, value=0)
    image = cv2.resize(image, (512, 512))
    
    one_hot = np.zeros((3, 512, 512), dtype=np.float32)
    one_hot[0, :, :] = (image == 1).astype(np.float32)
    
    # erode and dilate each organ, then select the biggest contour
    kernel = np.ones((3, 3), np.uint8)
    one_hot[0, :, :] = cv2.erode(one_hot[0, :, :], kernel, iterations=1)
    one_hot[0, :, :] = cv2.dilate(one_hot[0, :, :], kernel, iterations=1)
    
    one_hot[1, :, :] = (image == 2).astype(np.float32)
    one_hot[1, :, :] = cv2.erode(one_hot[1, :, :], kernel, iterations=1)
    one_hot[1, :, :] = cv2.dilate(one_hot[1, :, :], kernel, iterations=1)
    
    one_hot[2, :, :] = (image == 3).astype(np.float32)
    one_hot[2, :, :] = cv2.erode(one_hot[2, :, :], kernel, iterations=1)
    one_hot[2, :, :] = cv2.dilate(one_hot[2, :, :], kernel, iterations=1)
    
    image = one_hot
    image = (image - image.min()) / (image.max() - image.min())
    
    input = torch.tensor(image, dtype=torch.float32).to(config['device'])
    input = input.unsqueeze(0)  # Add batch dimension
    output = model(input)[0]
    
    for organ in organs:
        # Get indices for this organ
        if config["naive"]:
            idx_organ = organ_id == int(organ)
        else:            
            idx_organ = circ_organ_order[organ]
    
        organ_name = organ_dict[str(int(organ))]
        pred_organ = output[0, idx_organ].cpu().detach().numpy() * 512

        # fill the contour 
        blank = np.zeros((512, 512), dtype=np.float32)
        cv2.drawContours(blank, [pred_organ.astype(np.int32)], -1, 1, thickness=cv2.FILLED)
        
        dice = dice_score(blank, one_hot[int(organ)-1, :, :])
        aux = pd.DataFrame({'Image': [image_path], 'Organ': [organ_name], 'Dice': [dice], 'Unified': [not config['naive']]}).reset_index(drop=True)
        df = pd.concat([df, aux], ignore_index=True)



/home/ngaggion/miniconda3/envs/hybridgnet/lib/python3.13/site-packages/torch/nn/modules/instancenorm.py:115: UserWarning: input's size at dim=1 does not match num_features. You can silence this warning by not passing in num_features, which is not used because affine=False
  warnings.warn(
/tmp/ipykernel_1483307/174197709.py:62: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, aux], ignore_index=True)


In [13]:
df2 = df.copy()

In [ ]:
# concat df and df2
df_3 = pd.concat([df, df2], ignore_index=True)

# summarize the results
summary = df_3.groupby(['Organ', 'Unified']).agg({'Dice': ['mean', 'std']}).reset_index()
summary.columns = ['Organ', 'Unified', 'Mean Dice', 'Std Dice']
summary['Unified'] = summary['Unified'].apply(lambda x: 'Unified' if x else 'Non-Unified')
# round to 3 decimal places
summary['Mean Dice'] = summary['Mean Dice'].round(3)
summary['Std Dice'] = summary['Std Dice'].round(3)

df_gt_non_naive = pd.read_csv("../Results/CAMUS/CAMUS_atlas_seg_NN/results.csv")
df_gt_non_naive['Unified'] = True
df_gt_naive = pd.read_csv("../Results/CAMUS/CAMUS_atlas_seg/results.csv")
df_gt_naive['Unified'] = False
df_gt = pd.concat([df_gt_non_naive, df_gt_naive], ignore_index=True)
# replace organ with Organ in the columns
df_gt.rename(columns={'organ': 'Organ'}, inplace=True)

summary_gt = df_gt.groupby(['Organ', 'Unified']).agg({'dc': ['mean', 'std']}).reset_index()
summary_gt.columns = ['Organ', 'Unified', 'Mean Dice', 'Std Dice']
summary_gt['Mean Dice'] = summary_gt['Mean Dice'].round(3)
summary_gt['Std Dice'] = summary_gt['Std Dice'].round(3)
summary_gt['Unified'] = summary_gt['Unified'].apply(lambda x: 'Unified' if x else 'Non-Unified')
print(summary_gt)

     Organ      Unified  Mean Dice  Std Dice
0       LA  Non-Unified      0.986     0.004
1       LA      Unified      0.989     0.003
2  LV Endo  Non-Unified      0.989     0.002
3  LV Endo      Unified      0.990     0.002
4   LV Epi  Non-Unified      0.981     0.004
5   LV Epi      Unified      0.984     0.003


In [41]:
# Create mean ± std format for both summaries
summary['Dice_Score'] = summary['Mean Dice'].astype(str) + ' ± ' + summary['Std Dice'].astype(str)
summary_gt['Dice_Score'] = summary_gt['Mean Dice'].astype(str) + ' ± ' + summary_gt['Std Dice'].astype(str)

# Add source identifier to distinguish between the two datasets
summary['Input_Type'] = 'nnUNet_Masks'
summary_gt['Input_Type'] = 'Ground_Truth_Masks'

# Keep only the columns we need for the combined summary
summary_clean = summary[['Organ', 'Unified', 'Input_Type', 'Dice_Score']].copy()
summary_gt_clean = summary_gt[['Organ', 'Unified', 'Input_Type', 'Dice_Score']].copy()

# Combine the two summaries
combined_summary = pd.concat([summary_clean, summary_gt_clean], ignore_index=True)

# Option 2: Pivot to wide format for easier comparison
combined_wide = combined_summary.pivot_table(
    index=['Organ', 'Unified'], 
    columns='Input_Type', 
    values='Dice_Score',
    aggfunc='first'
).reset_index()

# Rename columns for clarity
combined_wide.rename(columns={
    'Ground_Truth_Masks': 'GT_Dice_Score',
    'nnUNet_Masks': 'nnUNet_Dice_Score'
}, inplace=True)

# Option 3: Create a side-by-side comparison table
print("\n=== Side-by-Side Comparison Table ===")
comparison_table = combined_wide.copy()
print(comparison_table)

# Option 4: Summary statistics with mean ± std format
print("\n=== Overall Performance Summary ===")
# Calculate overall statistics
gt_overall_mean = summary_gt['Mean Dice'].mean().round(3)
gt_overall_std = summary_gt['Std Dice'].mean().round(3)
nnunet_overall_mean = summary['Mean Dice'].mean().round(3)
nnunet_overall_std = summary['Std Dice'].mean().round(3)

overall_stats = pd.DataFrame({
    'Input_Type': ['Ground_Truth_Masks', 'nnUNet_Masks'],
    'Overall_Dice_Score': [
        f"{gt_overall_mean} ± {gt_overall_std}",
        f"{nnunet_overall_mean} ± {nnunet_overall_std}"
    ]
})
print(overall_stats)


=== Side-by-Side Comparison Table ===
Input_Type    Organ      Unified  GT_Dice_Score nnUNet_Dice_Score
0                LA  Non-Unified  0.988 ± 0.005     0.986 ± 0.004
1                LA      Unified   0.99 ± 0.004     0.989 ± 0.003
2           LV Endo  Non-Unified   0.99 ± 0.002     0.989 ± 0.002
3           LV Endo      Unified  0.991 ± 0.002      0.99 ± 0.002
4            LV Epi  Non-Unified  0.979 ± 0.005     0.981 ± 0.004
5            LV Epi      Unified  0.983 ± 0.003     0.984 ± 0.003

=== Overall Performance Summary ===
           Input_Type Overall_Dice_Score
0  Ground_Truth_Masks      0.987 ± 0.004
1        nnUNet_Masks      0.986 ± 0.003
